### Q1. Embedding a query

In [1]:
# Embed the following query:
# How does approximate nearest neighbor search work?
# The embedder returns a vector of 384 numbers. What's the first value (v[0])?

# -0.31
# -0.02
# 0.12
# 0.44

from embedder import Embedder
query = 'How does approximate nearest neighbor search work?'
embed = Embedder()

v1 = embed.encode(query)
print(" first value (v[0])=", round((v1[0]),2))

2026-06-30 19:13:55.805034956 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


 first value (v[0])= -0.02


### Q2. Cosine similarity

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [ ]:
# The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.

# Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, 
# embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?

# 0.07
# 0.37
# 0.68
# 0.92

sql_search_doc = [doc['content'] for doc in documents if doc['filename']=='02-vector-search/lessons/07-sqlitesearch-vector.md']
d1 = embed.encode(sql_search_doc[0]) 
print('cosine similarity =', round(v1.dot(d1),2))


cosine similarity = 0.36


In [4]:
sql_search_doc = [doc['content'] for doc in documents if doc['filename']=='02-vector-search/lessons/07-sqlitesearch-vector.md']

sql_search_doc

['# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done so far is e

### Q3. Chunking and search by hand

In [9]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
chunk_content = [chunk['content'] for chunk in chunks]
X = embed.encode_batch(chunk_content)
scores = X.dot(v1)


In [16]:
import numpy as np

location = np.argmax(-scores)
print("filename =", chunks[location]['filename'])

filename = 07-project-example/lessons/05-monitoring.md


### Q4. Vector search with minsearch

In [ ]:
from minsearch import Index
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, chunks)

In [ ]:
from minsearch import Index
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, chunks)

In [ ]:
query = 'What metric do we use to evaluate a search engine?'
query_vector = embed.encode(query)
result = vindex.search(query_vector, num_results=5)
print('Filename =', result[0]['filename'])

### Q5. Text search vs vector search

In [32]:
query = 'How do I store vectors in PostgreSQL?'
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(chunks)
result_text_search = index.search(query, num_results=5)
result_text_search

[{'start': 4000,
  'content': 'get 0.01.\n\nThe first score for `q1` vs `d` (0.32) is higher, so that query is more\nsimilar to the document about registration. The second score for `q2`\nvs `d` sits near 0, because installing Docker has nothing to do with\nregistration. A score near 0 means the two vectors are about as\ndifferent as they can be.\n\nThat\'s the whole idea behind vector search: similar texts get similar\nvectors, and a dot product tells us how similar.\n\n## Cosine similarity\n\nThe `all-MiniLM-L6-v2` model outputs normalized vectors - vectors with\nunit length. When both vectors are normalized, the dot product equals\ncosine similarity. That\'s why the model documentation says it "uses\ncosine similarity."\n\nCosine similarity measures the angle between two vectors, ignoring\ntheir length:\n\n- 1.0 = same direction (similar)\n- 0.0 = perpendicular (unrelated)\n- -1.0 = opposite direction (opposite meaning)\n\nFormally, if `theta` is the angle between two vectors, cosin

In [33]:
v_query = embed.encode(query)
result_vector_search = vindex.search(v_query, num_results=5)
result_vector_search

[{'start': 0,
  'content': '# Vector Search with PGVector\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=0P54MFyz-mc&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nMany real databases can do vector search. Elasticsearch has it, and\nthere are dedicated stores like Qdrant and Chroma. We\'ll go with\nPostgres. Most of us already run it at work, and the data engineering\ncourse uses it too. The concept is the same as with sqlitesearch, only\nthe database under the hood changes.\n\n[pgvector](https://github.com/pgvector/pgvector) is the PostgreSQL\nextension that makes this work. Install it and Postgres can do vector\nsimilarity search. On top of that you get the usual production features,\nlike concurrent access, transactions, and large datasets.\n\nWe\'ll run Postgres with pgvector in Docker.\n\n## Starting Postgres with pgvector\n\nPull the image and start a container:\n\n```bash\ndocker run -it \\\n    --name pgvector \\\n    -e POSTGRES_USER=user \\\n    -e POSTGRES_PASSWO

In [34]:
# Take the top 5 results from each method. 
# Which file shows up in the vector results but not in the text results?

# answer: 02-vector-search/lessons/08-pgvector.md

### Q6. Hybrid search

In [36]:

def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [ ]:

query = "How do I give the model access to tools?"
v_query = embed.encode(query)
vector_results = vindex.search(v_query, num_results=5)
text_results = index.search(query, num_results = 5)
results = rrf([vector_results, text_results])

In [38]:
results

[{'start': 4000,
  'content': ' function. `parameters` is a JSON schema\nfor the arguments, and we mark `query` as required so the model always\nfills it in.\n\n## Sending the question with the tool\n\nNow we send the same question as before, but this time we include the\ntool in the request:\n\n```python\nresponse = openai_client.responses.create(\n    model="gpt-5.4-mini",\n    input=messages,\n    tools=[search_tool],\n)\n\nresponse.output\n```\n\nLook at the output. Instead of a message with the answer, the response\ncontains a `function_call` entry. The model decided it needs to search\nthe FAQ before answering. Rather than reply, it asked us to run the\nsearch function first.\n\nLook at the arguments too. The model didn\'t pass our question\nverbatim. It judged the raw question wasn\'t the best query to search\nwith. So it rewrote our enrollment question into search keywords like\n"enroll late join course".\n\n## Executing the function and sending the result back\n\nThe function 

In [ ]:
# Which file is ranked first after RRF?

# 01-agentic-rag/lessons/01-intro.md
# 01-agentic-rag/lessons/13-function-calling.md
# 01-agentic-rag/lessons/14-agentic-loop.md
# 01-agentic-rag/lessons/16-other-frameworks.md

# Ans: 01-agentic-rag/lessons/13-function-calling.md